# APK 4720 / APK 6725

# Assignment # 9

Please submit your assignment as a Jupyter notebook (.ipynb file). Start a new Jupyter notebook and name it "YourName_Assignment_9"

Replace *YourName* with your first name and last names.


>BEFORE STARTING THE ASSIGNMENT, MAKE SURE THAT **ULTRALYTICS** IS INSTALLED IN YOUR ENVIRONMENT. 

Visit `docs.ultralytics.com/quickstart/` for instructions on how to install this package in your system.

Note, if you are using macOS, you will need to downgrade `numpy` to use ultralytics. Use the following command 
```
    !pip install "numpy<2"
```

## Learning Objectives

By the end of this assignment, students will be able to:

- Analyze gait data from video using pose estimation
- Detect gait events from videos
- Collect and process custom data


To complete this assignment, unzip the file `Assignment9.zip` included in GitHub. After unzipping the file, you will have access to a folder (`Assignment9`) that contains the models and data employed in this assignment. 

## Part A (2.5 points)

### Part A.1. (0.5 points)

In this section, you will load a gait video and process it using a pose estimation model. You will use the small version of the YOLO-pose family of models. If you prefer to use a smaller (lower accuracy, faster processing speed) or large (higher accuracy, lower processing speed), you can download the model from https://docs.ultralytics.com/tasks/pose/#models

1. Load the model `yolo26s-pose.pt` 
```Python
    from ultralytics import YOLO
    model_pose = YOLO(r'Assignment8/yolo26s-pose.pt')
```
2. Load the video `GaitVideo.mp4` and processes it with the model. 
```Python
    video_path = "Assignment9/GaitVideo.mp4"
    results = model.track(
        source=video_path,
        stream=True,
        tracker="bytetrack.yaml",
        persist=True,
        verbose=False,
        device = 'cpu'
        )
    results_list = list(results)
```

*Note*: If the kernel restarts by itself, try the following code instead
```Python
    video_path = "Assignment9/GaitVideo.mp4"
    results = model.predict(
        source=video_path,
        verbose=False,
        device = 'cpu'
        )
```


### Part A.2 (1 point)

The video presents a sagittal view of a person walking. 

Assuming that your results are stored in a variable called `results`, the following code shows the vertical position of the left and right ankles.

```Python
    import matplotlib.pyplot as plt
    from scipy.signal import savgol_filter
    #convert results to list for easier processing
    frame_ids = []
    pose_seq = []
    #put get measurements out of the results
    for i, r in enumerate(results_list):
        if r.keypoints is None or len(r.keypoints.xy) == 0:
            continue

        # Pick the highest-confidence person in frame
        kpts = r.keypoints.xy.cpu().numpy()  # (n_people, 17, 2)
        conf = r.boxes.conf.cpu().numpy() if r.boxes is not None else np.ones(len(kpts))

        person_idx = int(np.argmax(conf))

        frame_ids.append(i)
        pose_seq.append(kpts[person_idx])

    left_ankle_y = np.array(pose_seq)[:,[15],1]  # y-coordinate of left ankle
    right_ankle_y = np.array(pose_seq)[:,[16],1]
    # Apply Savitzky-Golay filter to smooth the signals
    right_ankle_y_smooth = savgol_filter(right_ankle_y.ravel(), 15, 3)
    left_ankle_y_smooth = savgol_filter(left_ankle_y.ravel(), 15, 3)
    plt.plot(frame_ids, right_ankle_y_smooth,label='Right Ankle')  # Plot y-coordinate of right ankle over time
    plt.plot(frame_ids, left_ankle_y_smooth,label='Left Ankle')  # Plot y-coordinate of left ankle over time
    plt.xlabel('Frame')
    plt.ylabel('Y-Coordinate')
    plt.title('Ankle Y-Coordinates Over Time')
    plt.legend()
    plt.show()
```

### Questions
1. How can you use this data (the vertical ankle position) to detect heel strike? 
2. Design an implement an algorithm for this purpose. Your algorithm should identify the frames corresponding to two heel-strike from the left foot and one for the right foot.


### Part A.3 (1 points)

The following code can be used to estimate the flexion/extension angle of the knee in the video 
```Python
    def angle_between(v1, v2):
        #calcular angle between two 2d-vectors
        v1_u = v1 / np.linalg.norm(v1, axis=1, keepdims=True)
        v2_u = v2 / np.linalg.norm(v2, axis=1, keepdims=True)
        dot_product = np.einsum('ij,ij->i', v1_u, v2_u)
        return np.degrees(np.arccos(dot_product))

    hip_left = np.array(pose_seq)[:,[11],:]  # left hip
    knee_left = np.array(pose_seq)[:,[13],:]  # left knee
    ankle_left = np.array(pose_seq)[:,[15],:]  # left ankle

    #vector hip to knee
    hip2kee_left = knee_left - hip_left
    #vector knee to ankle
    knee2ankle_left = ankle_left - knee_left

    knee_flexion_extension_left = angle_between(hip2kee_left.reshape(-1,2), knee2ankle_left.reshape(-1,2))
    knee_angle_left_smooth = savgol_filter(knee_flexion_extension_left, 15, 3)
    plt.plot(frame_ids, knee_angle_left_smooth,label='Left Knee Angle')
    plt.xlabel('Frame')
    plt.ylabel('Angle (degrees)')
    plt.title('Left Knee Angle Over Time')
    plt.legend()
```

### Questions
3. Plot the left knee angle between two heel-strikes. 
4. Compare the observed angle with results from the literature. What can you say about video-based estimated angle? 

# Part B (2.5 Points)

In this section, you will record a video of yourself walking similarly to the previous video. 

Make sure that:
1. Your full body is visible (from head to toes) so that the gait events can be properly localized.
2. The background is clear and there are no more people in the video.
3. You record at least 1 full step with each leg (from heel-strike to heel-strike).
4. Your video is recorded at 30 Frames Per Second (to prevent files that are too large).

Use the pose estimation model to process your video and plot the knee angle (left and right knees) between heel-strikes. 

# Extra (no value)

Record multiple steps per leg and compute the knee angle for each step. Normalize the measured angles in time and perform ensemble averaging. 